<a href="https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/AI%E6%8F%90%E5%95%8F%E8%88%87%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%AF%A6%E6%88%B0/03_%E5%95%86%E5%A0%B4%E5%AE%A2%E6%88%B6%E5%88%86%E7%BE%A4_%E9%9B%86%E7%BE%A4%E7%AF%87/%E5%95%86%E5%A0%B4%E5%AE%A2%E6%88%B6%E5%88%86%E7%BE%A4_%E9%9B%86%E7%BE%A4%E7%AF%87.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
# 若在 Google Colab 上執行，請先安裝中文字型支援庫與下載工具
%pip install wget mplfonts


# 🛍️ 範例 03：商場客戶價值分群 (集群篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本範例採用**「一段程式碼 + 一段 AI 互動學習指引」**的架構：
- 每一段程式碼都經過嚴格測試，保證可以直接執行。
- 每個程式儲存格的**正下方**，都緊接著一個專屬的文字儲存格，詳細解說程式原理，並提供依據核心觀念設計的 **AI 提問 Prompt**，讓你深入探討非監督學習、客群劃分與商業策略！

---

## 📊 一、資料集介紹 (`customer.csv`)

👉 **資料集直接下載**：[customer.csv (點此下載或於新分頁開啟)](https://raw.githubusercontent.com/roberthsu2003/machine_learning/main/AI%E6%8F%90%E5%95%8F%E8%88%87%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%AF%A6%E6%88%B0/03_%E5%95%86%E5%A0%B4%E5%AE%A2%E6%88%B6%E5%88%86%E7%BE%A4_%E9%9B%86%E7%BE%A4%E7%AF%87/customer.csv)

本數據集記錄了某大型百貨商場 200 位 VIP 會員的屬性與消費特徵：

| 欄位名稱 | 資料型態 | 數值範圍 | 業務意義與集群分析角色 |
| :--- | :--- | :--- | :--- |
| **性別** | 類別型 (`object`) | 男 / 女 | 會員生理性別（人口統計特徵） |
| **年齡** | 數值型 (`int64`) | 18 ~ 70 歲 | 會員年齡分佈 |
| **收入(千)** | 數值型 (`int64`) | 15 ~ 137 千元 | 會員年收入水平（千美元／千台幣） |
| **消費指數(1-100)** | 數值型 (`int64`) | 1 ~ 99 分 | 商場依消費頻率與總額評定之活躍分數（越接近 100 越愛買） |

> ⚠️ **重要本質差異**：本資料集**沒有任何「目標標籤 y」**！這屬於「非監督式學習 (Unsupervised Learning)」，目標是由演算法自動探索客群邊界。

---

## 🧭 二、機器學習全流程地圖

1. **環境配置與字型**：解決繪圖中文亂碼問題。
2. **資料載入與探索性分析 (EDA)**：觀察年收入與消費指數之二維分佈。
3. **特徵工程與標準化**：理解幾何歐氏距離為何必須透過 `StandardScaler` 消除尺度偏差。
4. **決定最佳分群數 K**：計算手肘法 (Elbow Method) 慣性曲線與輪廓係數 (Silhouette Score)。
5. **K-Means 演算法擬合與視覺化**：繪製 5 大客群散佈圖並標示群中心 (Centroids)。
6. **客群畫像萃取與實務推論**：為各群命名商業畫像，並為新進顧客即時匹配精準行銷策略。

---
### 【步驟 0】：環境準備與繁體中文字型設定
載入 Pandas、NumPy、Matplotlib、Seaborn 與 Scikit-Learn 集群套件，並設定繁體中文字型。

In [ ]:
# [步驟 0 程式碼] 載入套件與設定中文字型
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Microsoft JhengHei", "sans-serif"]
    plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！")


#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入集群分析所需套件，並設定 Matplotlib 中文字型，避免繪製客群分佈圖時文字標籤變亂碼。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **中文字型設定觀念**
   > 「在 Jupyter Notebook 中，Matplotlib 設定中文字型的底層邏輯是什麼？為什麼在 Colab 與本地端的字型設定方法可能不同？」


---
### 【步驟 1】：資料載入與探索性分析 (EDA)
讀取 `customer.csv`，檢視顧客年齡、收入與消費活躍分數的分佈狀態。

In [ ]:
# [步驟 1 程式碼] 資料載入與探索性資料分析
import os

if not os.path.exists("customer.csv"):
    import wget
    print("正在下載 customer.csv 資料集...")
    wget.download("https://raw.githubusercontent.com/roberthsu2003/machine_learning/main/AI%E6%8F%90%E5%95%8F%E8%88%87%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%AF%A6%E6%88%B0/03_%E5%95%86%E5%A0%B4%E5%AE%A2%E6%88%B6%E5%88%86%E7%BE%A4_%E9%9B%86%E7%BE%A4%E7%AF%87/customer.csv")
    print("\n✅ 下載完成！")

df = pd.read_csv("customer.csv")
print(f"資料維度: {df.shape} (共 {df.shape[0]} 位商場會員)")
display(df.head())

# 繪製原始二維分佈散佈圖 (收入 vs 消費指數)
plt.figure(figsize=(7, 5))
plt.scatter(df["收入(千)"], df["消費指數(1-100)"], color="royalblue", alpha=0.7, edgecolors="k")
plt.title("商場顧客原始二維分佈 (未分群前)", fontsize=13)
plt.xlabel("年收入 (千元)", fontsize=11)
plt.ylabel("消費指數 (1-100)", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  讀取 200 筆顧客數據。從未分群前的散佈圖中，肉眼已可隱約看出資料在二維平面上自然聚集為約 5 個群聚塊！

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **非監督式學習 (Unsupervised Learning) 本質**
   > 「這份資料沒有任何 `y`（目標標籤）。請用白話文向我解釋：什麼是非監督式學習？它和先前的迴歸與分類任務有何本質上的不同？」

2. 📌 **切分測試集的迷思**
   > 「在非監督式分群中，初學者常問『為什麼不需要切分 80% 訓練集和 20% 測試集？』請解釋在純探索性分群中為什麼不需要切分測試集？」

3. 🛠️ **進階探索改寫 Prompt**
   > 「請幫我寫一段程式碼，利用 Seaborn 的 `pairplot` 一次繪製『年齡』、『收入』與『消費指數』三者之間的兩兩散佈圖與對角線分佈直方圖。」


---
### 【步驟 2】：特徵工程與標準化 (StandardScaler)
K-Means 演算法是以「歐氏幾何距離」計算相似度。如果不做特徵標準化，數值範圍大的特徵（如年收入 15~137）將會壓制數值範圍小的特徵（如年齡 18~70）。

In [ ]:
# [步驟 2 程式碼] 特徵萃取與 StandardScaler 標準化
from sklearn.preprocessing import StandardScaler

features = ["收入(千)", "消費指數(1-100)"]
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"✅ 特徵標準化完成！標準化後均值: {X_scaled.mean(axis=0).round(2)}, 標準差: {X_scaled.std(axis=0).round(2)}")


#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將收入與消費指數標準化為均值為 0、標準差為 1 的無維度數值，保證幾何距離計算的公正性。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **幾何歐氏距離與尺度偏差**
   > 「K-Means 是基於歐氏距離 (Euclidean Distance) 運作的。如果一個欄位數值動輒上萬，另一個欄位只有 1 到 10，為什麼不做標準化會導致集群結果完全失真？」

2. 📌 **StandardScaler vs MinMaxScaler**
   > 「在分群問題中，`StandardScaler`（Z-score 標準化）與 `MinMaxScaler`（縮放到 0~1）各自適用於什麼場景？面對離群值時誰更具穩健性？」


---
### 【步驟 3】：決定最佳群數 K (手肘法 Elbow Method 與輪廓係數 Silhouette Score)
在非監督學習中，演算法不知道到底要分幾群。我們必須透過「慣性 (Inertia)」與「輪廓係數」雙重客觀指標找出最佳 K 值。

In [ ]:
# [步驟 3 程式碼] 手肘法與輪廓係數計算
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_range = range(2, 11)
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, init="k-means++", random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

# 雙圖評估
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# 1. 手肘法曲線 (Inertia)
axes[0].plot(k_range, inertias, "bo-", lw=2, markersize=7)
axes[0].axvline(x=5, color="red", linestyle="--", label="最佳手肘拐點 (K=5)")
axes[0].set_title("手肘法 (Elbow Method) 慣性曲線", fontsize=12)
axes[0].set_xlabel("群數 K")
axes[0].set_ylabel("群內平方和 (Inertia)")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# 2. 輪廓係數曲線 (Silhouette Score)
axes[1].plot(k_range, sil_scores, "rs-", lw=2, markersize=7)
axes[1].axvline(x=5, color="red", linestyle="--", label="高輪廓峰值 (K=5)")
axes[1].set_title("各 K 值之輪廓係數 (Silhouette Score)", fontsize=12)
axes[1].set_xlabel("群數 K")
axes[1].set_ylabel("平均輪廓係數 (越接近 1 越好)")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

print(f"✅ 最佳群數判定：K=5 時輪廓係數高達 {sil_scores[3]:.3f}，手肘拐點最顯著！")


#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  我們嘗試了 K=2 到 K=10。手肘圖在 K=5 處出現明顯的急遽趨緩拐點，且 K=5 的輪廓係數高達 0.555，證明 K=5 是統計學上最具凝聚力與分離度的最佳群數！

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **手肘法 (Elbow Method) 原理**
   > 「手肘法中的『慣性 (Inertia / WCSS)』數學定義是什麼？為什麼當 K 越大慣性一定會越小？我們該如何客觀辨識哪一個點才是『手肘拐點』？」

2. 📌 **輪廓係數 (Silhouette Score) 深刻剖析**
   > 「輪廓係數的數值範圍介於多少之間？+1、0、-1 各代表什麼幾何意義？群內緊密距 $a(i)$ 與群外分離距 $b(i)$ 是如何構成輪廓分數的？」

3. 🛠️ **進階探索改寫 Prompt**
   > 「如果我把『年齡』特徵也納入分群（三維分群），手肘法的最佳 K 值會改變嗎？請幫我修改代碼測試包含年齡的三維分群手肘圖。」


---
### 【步驟 4】：K-Means 演算法擬合與分群散佈圖視覺化
以 K=5 正式訓練模型，將 200 位顧客劃分至 5 個群體，並繪製色彩分明的客群散佈圖與群中心 (Centroids)。

In [ ]:
# [步驟 4 程式碼] K-Means (K=5) 擬合與二維視覺化
kmeans = KMeans(n_clusters=5, init="k-means++", random_state=42, n_init=10)
df["客群標籤"] = kmeans.fit_predict(X_scaled)

# 將群中心逆轉換回原始真實數值尺度
centroids = scaler.inverse_transform(kmeans.cluster_centers_)

colors = ["#e11d48", "#2563eb", "#16a34a", "#d97706", "#7c3aed"]
cluster_names = [
    "Cluster 0: 高收高消費 VIP",
    "Cluster 1: 大眾中堅主力",
    "Cluster 2: 高收保守高資產",
    "Cluster 3: 年輕潮流衝動型",
    "Cluster 4: 平價精打細算型"
]

plt.figure(figsize=(9, 6))
for c_id in range(5):
    sub = df[df["客群標籤"] == c_id]
    plt.scatter(sub["收入(千)"], sub["消費指數(1-100)"], s=60, color=colors[c_id], label=f"群體 #{c_id} ({len(sub)}人)", alpha=0.8, edgecolors="k")

# 標記群中心 (Centroids)
plt.scatter(centroids[:, 0], centroids[:, 1], s=250, c="yellow", marker="*", edgecolors="black", lw=1.5, label="群中心 (Centroids)", zorder=10)

plt.title("商場顧客 5 大價值客群分群散佈圖 (K-Means)", fontsize=14)
plt.xlabel("年收入 (千元)", fontsize=12)
plt.ylabel("商場消費活躍指數 (1-100)", fontsize=12)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  我們將 200 位顧客劃分成 5 個顏色鮮明的客群，黃色星星標示了各群的核心代表（群中心）。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **K-Means 演算法底層迭代步驟**
   > 「K-Means 演算法是如何透過『隨機初始化群中心 $\rightarrow$ 計算最近距離歸類 $\rightarrow$ 重新計算幾何重心 $\rightarrow$ 迭代直到收斂』的過程完成分群的？」

2. 📌 **k-means++ 初始化優勢**
   > 「參數中的 `init=k-means++` 是什麼意思？它比早期的純隨機初始化群中心解決了什麼局部極小值 (Local Minima) 的問題？」

3. 🛠️ **進階探索改寫 Prompt**
   > 「請幫我改用另一種分群演算法：『階層分群法 (Hierarchical Clustering / AgglomerativeClustering)』，並繪製樹狀圖 (Dendrogram)。」


---
### 【步驟 5】：客群畫像分析與新顧客分群推論
統計各群體的平均收入與消費活躍度，萃取商業標籤，並為新加入商場的會員即時匹配所屬客群與行銷策略。

In [ ]:
# [步驟 5 程式碼] 客群統計畫像與新顧客分群推論
profile_df = df.groupby("客群標籤")[["收入(千)", "消費指數(1-100)", "年齡"]].mean().round(1)
counts = df["客群標籤"].value_counts().sort_index()
profile_df["人數佔比"] = [(counts[i] / len(df) * 100) for i in range(5)]
profile_df["人數佔比"] = profile_df["人數佔比"].map(lambda x: f"{x:.1f}%")
display(profile_df)

# 建立推論函式 (使用 DataFrame 傳遞維持特徵欄位名稱)
def classify_new_customer(scaler_obj, kmeans_model, income, score):
    sample_df = pd.DataFrame([{"收入(千)": income, "消費指數(1-100)": score}])
    scaled_point = scaler_obj.transform(sample_df)
    c_id = int(kmeans_model.predict(scaled_point)[0])
    return c_id

# 測試新進會員推論
test_cases = [
    ("會員 A (高收高買貴客)", 95, 88),
    ("會員 B (年輕衝動潮流族)", 22, 75),
    ("會員 C (高收低消省錢族)", 100, 16)
]

print("=== 新會員即時分群推論測試 ===")
for name, inc, sc in test_cases:
    cid = classify_new_customer(scaler, kmeans, inc, sc)
    print(f"{name:22s} [收入:{inc}k, 指數:{sc}] -> 歸類至群體 #{cid}")


#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  計算各客群的平均指標，並建立新顧客快速分群函式。商場可依據不同群體的特徵，給予專屬的促銷活動（例如對高收高買群體推動私人管家預約、對高收低買群體推播高保值精品）。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **客群畫像 (Customer Profiling) 商業價值**
   > 「在商業行銷 (CRM) 實務中，我們如何根據這 5 大客群的收入與消費特徵，為每一群客戶設計差異化的促銷檔期與忠誠度獎勵制度？」

2. 📌 **新數據即時推論 (Inference on New Data)**
   > 「當有新會員加入時，K-Means 是如何透過已固定下來的群中心 (Centroids) 迅速判定其所屬群體的？需要重新 fit 整個資料集嗎？」

3. 🛠️ **進階探索改寫 Prompt**
   > 「我想將這個分群推論模型製作成一個獨立的桌面視窗軟體 (Tkinter GUI)，提供滑桿調整收入與消費指數，並即時彈出客群標籤與行銷建議，該如何寫？」
